# Project: Cozy Bean — Retail Operations & Analytics Dashboard
### Exploratory Data Analysis: Week 5 Milestone

**Author:** Fiteh Tesfaye 

**Project Overview:** This project acts as an executive-level Exploratory Data Analysis (EDA) dashboard built around the operational logs of **Cozy Bean** across four unique commercial environments (Airport, Mall, Suburb, and City Center). The analysis moves from raw column profiling into high-dimensional grouping, multi-variable data reshaping, and dynamic outlier auditing.

**Skills Demonstrated:**
* **High-Dimensional Column Summarization** (`.describe()`, `.value_counts()`, `pd.crosstab`)
* **Advanced Multi-Variable Grouping & Named Aggregations** (`.groupby()`, `.agg()`)
* **Data Shape Restructuring & Transformations** (`pivot_table()`, `melt()`)
* **Vectorized Window Operations** (`.transform()`)

## Data Initialization (Code Cell)

In [1]:
import numpy as np
import pandas as pd

# Set seed for absolute calculation consistency
np.random.seed(42)

# Generate 30 days of records across 4 unique storefront properties (120 observations)
dates = pd.date_range(start="2026-04-01", end="2026-04-30").repeat(4)
stores = ["A", "B", "C", "D"] * 30
locations = {"A": "City Center", "B": "Airport", "C": "Mall", "D": "Suburb"}
store_locations = [locations[s] for s in stores]

df = pd.DataFrame({
    'date': dates,
    'store': stores,
    'location': store_locations
})

# Engineer time intelligence components 
df['weekday'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6])

# Construct synthetic environmental noise 
weather_options = ['Sunny', 'Rainy', 'Cloudy', 'Unknown']
df['weather'] = np.random.choice(weather_options, size=len(df), p=[0.45, 0.35, 0.17, 0.03])

# Apply customized baseline volume floors mapped directly to localized consumer patterns
base_volumes = {'A': 120, 'B': 95, 'C': 160, 'D': 80}
df['drinks_sold'] = df['store'].map(base_volumes) + np.random.normal(0, 15, size=len(df))

# Account for weekend traffic expansion
df.loc[df['is_weekend'], 'drinks_sold'] += np.random.uniform(20, 45, size=df['is_weekend'].sum())
df['drinks_sold'] = df['drinks_sold'].round(0).astype(float)

# Generate adaptive dynamically calculated pricing variations
df['price_per_cup'] = np.where(df['is_weekend'], 
                               np.random.uniform(3.80, 4.20, size=len(df)),
                               np.random.uniform(3.20, 3.60, size=len(df)))
df['price_per_cup'] = df['price_per_cup'].round(2)
df['revenue'] = (df['drinks_sold'] * df['price_per_cup']).round(2)

print(f"Dataset successfully initialized. Matrix shape: {df.shape}")
df.head()

Dataset successfully initialized. Matrix shape: (120, 9)


,date,store,location,weekday,is_weekend,weather,drinks_sold,price_per_cup,revenue
0,2026-04-01,A,City Center,Wednesday,False,Sunny,109.0,3.26,355.34
1,2026-04-01,B,Airport,Wednesday,False,Cloudy,90.0,3.50,315.00
2,2026-04-01,C,Mall,Wednesday,False,Rainy,154.0,3.45,531.30
3,2026-04-01,D,Suburb,Wednesday,False,Rainy,58.0,3.24,187.92
4,2026-04-02,A,City Center,Thursday,False,Sunny,124.0,3.23,400.52


## Section 1: Structural Exploration & Column Distribution Profiling
Before deep-diving into specific performance answers, we audit the baseline structural attributes and distributions of our variables.

In [2]:
print("--- Summary Distribution of Numeric Elements ---")
display(df.describe())

print("\n--- Categorical Volume Densities (Weather Dynamics) ---")
display(df['weather'].value_counts())

print("\n--- Complete Global Structural Analysis (Transposed) ---")
display(df.describe(include='all').T)

--- Summary Distribution of Numeric Elements ---


,date,drinks_sold,price_per_cup,revenue
count,120,120.000000,120.000000,120.000000
mean,2026-04-15 12:00:00,123.066667,3.573833,443.380917
min,2026-04-01 00:00:00,58.000000,3.200000,187.920000
25%,2026-04-08 00:00:00,96.750000,3.365000,333.622500
50%,2026-04-15 12:00:00,121.000000,3.495000,423.605000
75%,2026-04-23 00:00:00,145.500000,3.815000,531.667500
max,2026-04-30 00:00:00,231.000000,4.180000,965.580000
std,NaN,36.078750,0.292722,149.101060



--- Categorical Volume Densities (Weather Dynamics) ---


weather
Sunny      58
Rainy      38
Cloudy     23
Unknown     1
Name: count, dtype: int64


--- Complete Global Structural Analysis (Transposed) ---


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
date,120,NaN,NaN,NaN,2026-04-15 12:00:00,2026-04-01 00:00:00,2026-04-08 00:00:00,2026-04-15 12:00:00,2026-04-23 00:00:00,2026-04-30 00:00:00,NaN
store,120,4,A,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,120,4,City Center,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weekday,120,7,Wednesday,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_weekend,120,2,False,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weather,120,4,Sunny,58,NaN,NaN,NaN,NaN,NaN,NaN,NaN
drinks_sold,120.0,NaN,NaN,NaN,123.066667,58.0,96.75,121.0,145.5,231.0,36.07875
price_per_cup,120.0,NaN,NaN,NaN,3.573833,3.2,3.365,3.49375,3.815,4.18,0.292722
revenue,120.0,NaN,NaN,NaN,443.380917,187.92,333.6225,423.605,531.6675,965.58,149.10106


## Section 2: Regional Performance Slicing
To keep tables looking presentation-ready and prevent hard-to-index multi-level hierarchical columns, we use explicit named aggregations within a grouped performance summary matrix.

In [3]:
executive_store_summary = df.groupby(['store', 'location']).agg(
    total_units_sold  = ('drinks_sold', 'sum'),
    mean_daily_sales  = ('drinks_sold', 'mean'),
    max_single_day    = ('drinks_sold', 'max'),
    total_gross_rev   = ('revenue', 'sum'),
    mean_ticket_price = ('price_per_cup', 'mean')
).round(2).sort_values(by='total_gross_rev', ascending=False)

print("Executive Regional Store Profiles:")
executive_store_summary

Executive Regional Store Profiles:


,,total_units_sold,mean_daily_sales,max_single_day,total_gross_rev,mean_ticket_price
store,location,,,,,
C,Mall,5052.0,168.40,231.0,18255.30,3.59
A,City Center,3816.0,127.20,170.0,13642.04,3.55
B,Airport,3206.0,106.87,153.0,11521.40,3.57
D,Suburb,2694.0,89.80,142.0,9786.97,3.59


## Section 3: Data Reshaping (Wide vs. Long Formats)
We transition between structural states optimized for presentation engines (wide matrices via `pivot_table`) and configurations built for processing downstream visualization tools (long linear tables via `melt`).

In [4]:
print("--- Pivot Table Matrix: Location Volume Across Weather Environments ---")
wide_weather_pivot = df.pivot_table(
    index='location',
    columns='weather',
    values='drinks_sold',
    aggfunc='mean'
).round(1)
display(wide_weather_pivot)

print("\n--- Melting Wide Matrix View back into a Tidy Format ---")
long_tidy_format = wide_weather_pivot.reset_index().melt(
    id_vars='location',
    var_name='weather_condition',
    value_name='average_drinks_sold'
)
display(long_tidy_format.head(8))

print("\n--- Cross-Tabulation: Operational Log Volumes Across Weekdays ---")
display(pd.crosstab(df['weekday'], df['weather']))

--- Pivot Table Matrix: Location Volume Across Weather Environments ---


weather,Cloudy,Rainy,Sunny,Unknown
location,,,,
Airport,105.8,102.3,108.1,140.0
City Center,123.0,121.0,132.5,NaN
Mall,153.5,168.3,171.9,NaN
Suburb,84.6,93.2,89.8,NaN



--- Melting Wide Matrix View back into a Tidy Format ---


,location,weather_condition,average_drinks_sold
0,Airport,Cloudy,105.8
1,City Center,Cloudy,123.0
2,Mall,Cloudy,153.5
3,Suburb,Cloudy,84.6
4,Airport,Rainy,102.3
5,City Center,Rainy,121.0
6,Mall,Rainy,168.3
7,Suburb,Rainy,93.2



--- Cross-Tabulation: Operational Log Volumes Across Weekdays ---


weather,Cloudy,Rainy,Sunny,Unknown
weekday,,,,
Friday,2,8,6,0
Monday,2,5,9,0
Saturday,2,3,10,1
Sunday,1,7,8,0
Thursday,8,3,9,0
Tuesday,4,5,7,0
Wednesday,4,7,9,0


## Section 4: Operational Challenges Resolved
This section isolates key business intelligence concerns using sophisticated vectorized window operations (`.transform()`).

In [5]:
print("=== Challenge 1: Find the (Weekday, Weather) Matrix Intersection with Busiest Mean Revenue ===")
top_performing_matrix = df.groupby(['weekday', 'weather']).agg(
    avg_daily_revenue = ('revenue', 'mean'),
    total_sample_days = ('revenue', 'count')
).round(2).sort_values(by='avg_daily_revenue', ascending=False)
display(top_performing_matrix.head(3))

print("\n=== Challenge 2: Compute Store Market Shares of Total Gross Revenue ===")
total_portfolio_revenue = df['revenue'].sum()
store_market_shares = df.groupby('store')['revenue'].sum() / total_portfolio_revenue
store_market_shares_df = (store_market_shares * 100).round(2).to_frame(name='market_share_percentage')
display(store_market_shares_df.sort_values(by='market_share_percentage', ascending=False))

print("\n=== Challenge 3: Flag Operational Exceptions Using Store-Specific Boundaries ===")
# Broadcast moving statistics back onto historical records via .transform()
df['store_mean'] = df.groupby('store')['drinks_sold'].transform('mean')
df['store_std']  = df.groupby('store')['drinks_sold'].transform('std')

# Flag entries exceeding the standard statistical exception line: Mean + 2 Standard Deviations
df['is_outlier'] = df['drinks_sold'] > (df['store_mean'] + (2 * df['store_std']))

outliers_detected = df[df['is_outlier']][['date', 'store', 'location', 'drinks_sold', 'store_mean', 'store_std']]
print(f"Successfully isolated {len(outliers_detected)} variance outlier alerts:")
display(outliers_detected)

=== Challenge 1: Find the (Weekday, Weather) Matrix Intersection with Busiest Mean Revenue ===


,,avg_daily_revenue,total_sample_days
weekday,weather,,
Saturday,Rainy,652.28,3
Sunday,Sunny,639.94,8
Saturday,Sunny,586.16,10



=== Challenge 2: Compute Store Market Shares of Total Gross Revenue ===


,market_share_percentage
store,
C,34.31
A,25.64
B,21.65
D,18.39



=== Challenge 3: Flag Operational Exceptions Using Store-Specific Boundaries ===
Successfully isolated 8 variance outlier alerts:


,date,store,location,drinks_sold,store_mean,store_std
14,2026-04-04,C,Mall,231.0,158.800000,24.085480
43,2026-04-11,D,Suburb,138.0,87.666667,22.912728
46,2026-04-12,C,Mall,213.0,158.800000,24.085480
70,2026-04-18,C,Mall,217.0,158.800000,24.085480
72,2026-04-19,A,City Center,170.0,127.200000,16.607850
75,2026-04-19,D,Suburb,142.0,87.666667,22.912728
99,2026-04-25,D,Suburb,137.0,87.666667,22.912728
117,2026-04-30,B,Airport,153.0,106.333333,20.011606


## Unified Reporting Dashboard (Code Cell)

In [6]:
day_sequence = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

print("=" * 80)
print("                 COZY BEAN ENTERPRISE PORTFOLIO SYSTEM DASHBOARD              ")
print("=" * 80)
print(f"Reporting Timeline Horizon   : {df['date'].nunique()} Logging Days Covered")
print(f"Active Commercial Footprint  : {df['store'].nunique()} Global Operating Stores")
print(f"Aggregated System Volumetrics: {int(df['drinks_sold'].sum()):,} Cups Dispatched")
print(f"Total System Gross Income    : ${df['revenue'].sum():,.2f}")
print(f"Portfolio Mean Product Price : ${df['price_per_cup'].mean():.2f} per Unit")
print(f"Identified Variance Alarms   : {df['is_outlier'].sum()} Outlier Events Registered")
print("-" * 80)
print("\nSTORE METRIC ACCOUNTING SUMMARY:")
print(df.groupby('store').agg(
    total_units=('drinks_sold', 'sum'),
    gross_revenue=('revenue', 'sum'),
    daily_mean_units=('drinks_sold', 'mean'),
    variance_anomalies=('is_outlier', 'sum')
).round(1).to_string())

print("\n" + "-" * 80)
print("WEEKDAY REVENUE MATRIX BY DISTRIBUTION POINT:")
weekly_revenue_pivot = df.pivot_table(
    index='weekday', 
    columns='store', \
    values='revenue', 
    aggfunc='sum'
).reindex(day_sequence).round(0)
print(weekly_revenue_pivot.to_string())
print("=" * 80)

                 COZY BEAN ENTERPRISE PORTFOLIO SYSTEM DASHBOARD              
Reporting Timeline Horizon   : 30 Logging Days Covered
Active Commercial Footprint  : 4 Global Operating Stores
Aggregated System Volumetrics: 14,768 Cups Dispatched
Total System Gross Income    : $53,205.71
Portfolio Mean Product Price : $3.57 per Unit
Identified Variance Alarms   : 8 Outlier Events Registered
--------------------------------------------------------------------------------

STORE METRIC ACCOUNTING SUMMARY:
       total_units  gross_revenue  daily_mean_units  variance_anomalies
store                                                                  
A           3816.0        13642.0             127.2                   1
B           3206.0        11521.4             106.9                   1
C           5052.0        18255.3             168.4                   3
D           2694.0         9787.0              89.8                   3

------------------------------------------------------------

## Portfolio Performance Insights
* **Revenue Driver:** The **Mall location (Store C)** is the dominant performer, commanding the highest total monthly volume and market share.
* **Volatility Analysis:** The use of `.transform()` successfully uncovered abnormal peaks in demand that fall outside the standard two standard-deviation operating line.
* **Structural Reshaping:** Switching patterns from flat data tables to multidimensional summary indices revealed unique variations in weekend metrics.

